# 🎼 VIRASAT AI — Phase 1: Extraction Lab

> **Objective**: Extract clean, isolated vocal stems from heritage Pakistani music recordings using Demucs v4. Evaluate quality using SDR, SIR, SAR, and the custom Virasat Score.

This notebook is designed to be run in **Google Colab** with a GPU runtime for maximum speed.

## 1. Setup Environment

In [ ]:
# Install core dependencies
!pip install -q demucs librosa soundfile mir_eval matplotlib numpy scipy yt-dlp

# Clone repository (if running in Colab)
import os
if not os.path.exists('VIRASAT_AI'):
    # Note: Replace with actual repo URL when public
    !git clone https://github.com/code-with-idrees/VIRASAT_AI.git
    %cd VIRASAT_AI
else:
    %cd VIRASAT_AI

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## 2. Download Test Songs
Using our `download_songs.py` script to fetch audio from YouTube.

In [ ]:
!python phase1_extraction_lab/scripts/download_songs.py --config phase1_extraction_lab/config/test_songs.json

## 3. Run Demucs v4 Stem Separation
We test both `htdemucs` (general purpose) and `htdemucs_ft` (fine-tuned) models.

In [ ]:
# Run separation using the fine-tuned model
!python phase1_extraction_lab/scripts/stem_separator.py \
    --input phase1_extraction_lab/data/raw/ \
    --model htdemucs_ft \
    --output phase1_extraction_lab/data/stems/

## 4. Quality Analysis (SDR, SIR, SAR, Virasat Score)

In [ ]:
# Analyze all extracted vocal stems
!python phase1_extraction_lab/scripts/quality_metrics.py \
    --estimated phase1_extraction_lab/data/stems/htdemucs_ft/

## 5. Instrument Bleed Detection
Specially identifying Sitar, Harmonium, and Tabla bleed in the vocal track.

In [ ]:
# Run bleed detection and generate spectrogram reports
!python phase1_extraction_lab/scripts/bleed_detector.py \
    --input phase1_extraction_lab/data/stems/htdemucs_ft/ \
    --report

## 6. View Results

In [ ]:
import IPython.display as ipd
from IPython.display import display, Image
import glob

print("🎵 Enhanced Vocals (Noor Jehan example):")
try:
    vocal_files = glob.glob("phase1_extraction_lab/data/stems/htdemucs_ft/**/vocals.wav", recursive=True)
    if vocal_files:
        display(ipd.Audio(vocal_files[0]))
except Exception as e:
    print("File not found or not processed yet.")

print("\n📊 Bleed Spectrogram:")
try:
    report_files = glob.glob("phase1_extraction_lab/data/reports/*.png")
    if report_files:
        display(Image(filename=report_files[0]))
except Exception as e:
    print("Spectrogram not generated yet.")